# UniMiSS+ Fine-tuning — Local Machine

Notebook chạy trên máy local. Chạy **tuần tự từ trên xuống**.

## 1. Kiểm tra môi trường

In [1]:
import os, sys, torch
print("Python    :", sys.executable)
print("PyTorch   :", torch.__version__)
print("CPU cores :", os.cpu_count())
print("CUDA      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))


Python    : /home/pc/venvs/torch/bin/python
PyTorch   : 2.12.0+cu130
CPU cores : 16
CUDA      : True
GPU       : NVIDIA GeForce RTX 5060 Ti


## 2. Cài dependencies còn thiếu

Bỏ qua nếu đã cài rồi.

In [2]:
# !pip install onnx "onnxruntime-gpu[cuda,cudnn]==1.26.0" nibabel scipy scikit-learn tqdm -q


In [3]:
import onnxruntime as ort
print("ORT version :", ort.__version__)
print("Providers   :", ort.get_available_providers())


ORT version : 1.26.0
Providers   : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


## 3. Cấu hình đường dẫn

Chỉ cần chỉnh ở đây, các cell bên dưới tự dùng biến này.

In [4]:
import os, sys

SCRIPT     = "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/Downstream/2D/Cls/main_flexible.py"
COVID_ROOT = "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset"
QUEX_ROOT  = "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-QU-EX/Lung Segmentation Data/Lung Segmentation Data"
CHEST_XRAY = "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/ChestX-ray"
PRETRAIN   = "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/UniMissPlus.pth"
RESULTS    = "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results"

os.makedirs(RESULTS, exist_ok=True)

checks = {
    "Script"   : SCRIPT,
    "COVID"    : COVID_ROOT,
    "QUEX"     : QUEX_ROOT,
    "ChestXray": CHEST_XRAY,
    "Pretrain" : PRETRAIN,
}
for name, path in checks.items():
    status = "✓" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"{name:10}: {status}  {path}")


Script    : ✓  /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/Downstream/2D/Cls/main_flexible.py
COVID     : ✓  /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset
QUEX      : ✓  /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-QU-EX/Lung Segmentation Data/Lung Segmentation Data
ChestXray : ✓  /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/ChestX-ray
Pretrain  : ✓  /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/UniMissPlus.pth


---
# Train COVID-19 Radiography Dataset
## 4a. Train — 3 seeds

In [9]:
!python Downstream/2D/Cls/main_flexible.py \
  --task covid \
  --covid_mode other_normal \
  --covid_root "{COVID_ROOT}" \
  --output_dir "{RESULTS}/covid_other_normal" \
  --pre_train \
  --pre_train_path "{PRETRAIN}" \
  --gpu "0" \
  --batch_size 128 \
  --test_batch_size 128 \
  --num_workers 8 \
  --epochs 40 \
  --learning_rate 0.00005 \
  --seeds "42"


=== Running seed 42 -> /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/covid_other_normal/seed_42 ===
Loaded 16933 COVID other-normal train images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset
Loaded 4232 COVID other-normal test images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset
  + Number of Encoder Params: 14.77(e6)
  + Number of Network Params: 14.77(e6)
Loaded 198 pretrained layers from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/UniMissPlus.pth
eval: 100%|█████████████████████████████████████| 34/34 [00:07<00:00,  4.64it/s]
epoch=0 | train_loss=0.615156 | lr=4.88821e-05 | accuracy=0.870038 | macro_auc=0.959428 | macro_ap=0.92242
eval: 100%|█████████████████████████████████████| 34/34 [00:07<00:00,  4.61it/s]
epoch=1 | train_loss=0.430168 | lr=4.77528e-05 | accuracy=0.892486 | macro_auc=

## 4b. Eval + Export ONNX + Grad-CAM

In [8]:
!python Downstream/2D/Cls/main_flexible.py \
  --task covid \
  --covid_mode other_normal \
  --covid_root "{COVID_ROOT}" \
  --output_dir "{RESULTS}/covid_other_normal_eval" \
  --checkpoint_path "{RESULTS}/covid_other_normal/best.pth" \
  --gpu "0" \
  --test_batch_size 128 \
  --num_workers 8 \
  --eval_only \
  --export_onnx \
  --compare_onnx \
  --onnx_runtime_provider cuda \
  --onnx_batch_size 0 \
  --benchmark_warmup_batches 2 \
  --grad_cam \
  --grad_cam_per_class \
  --grad_cam_per_class_samples 1


Loaded 16933 COVID other-normal train images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset
Loaded 4232 COVID other-normal test images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset
  + Number of Encoder Params: 14.77(e6)
  + Number of Network Params: 14.77(e6)
Loaded model weights from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/covid_other_normal/best.pth
eval: 100%|█████████████████████████████████████| 34/34 [00:08<00:00,  4.16it/s]
accuracy=0.965737 | macro_auc=0.995976 | macro_ap=0.993422
/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/Downstream/2D/Cls/main_flexible.py:250: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.

## 4c. Cross-eval: model COVID-19 Radiography → QU-Ex

In [ ]:
!python Downstream/2D/Cls/main_flexible.py \
  --task covid_qu_ex \
  --covid_qu_ex_root "{QUEX_ROOT}" \
  --output_dir "{RESULTS}/eval_covid19_to_quex" \
  --checkpoint_path "{RESULTS}/covid_other_normal/best.pth" \
  --gpu "0" \
  --test_batch_size 128 \
  --num_workers 8 \
  --eval_only


---
# Train COVID-QU-Ex
## 5a. Train — 3 seeds

In [11]:
!python Downstream/2D/Cls/main_flexible.py \
  --task covid_qu_ex \
  --covid_qu_ex_root "{QUEX_ROOT}" \
  --output_dir "{RESULTS}/train_quex" \
  --pre_train \
  --pre_train_path "{PRETRAIN}" \
  --gpu "0" \
  --batch_size 128 \
  --test_batch_size 128 \
  --num_workers 8 \
  --epochs 40 \
  --learning_rate 0.00005 \
  --seeds "42"


=== Running seed 42 -> /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/train_quex/seed_42 ===
Loaded 27132 COVID-QU-Ex train images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-QU-EX/Lung Segmentation Data/Lung Segmentation Data
Loaded 6788 COVID-QU-Ex test images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-QU-EX/Lung Segmentation Data/Lung Segmentation Data
  + Number of Encoder Params: 14.77(e6)
  + Number of Network Params: 14.77(e6)
Loaded 198 pretrained layers from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/UniMissPlus.pth
eval: 100%|█████████████████████████████████████| 54/54 [00:11<00:00,  4.64it/s]
epoch=0 | train_loss=0.629966 | lr=4.88789e-05 | accuracy=0.876547 | macro_auc=0.975398 | macro_ap=0.957012
eval: 100%|█████████████████████████████████████| 54/54 [00:11<00:00,  4.79it/s]
epoch=1 | train_loss=0.455424 | lr=4.77496

## 5b. Eval + Export ONNX + Grad-CAM

In [9]:
!python Downstream/2D/Cls/main_flexible.py \
  --task covid_qu_ex \
  --covid_qu_ex_root "{QUEX_ROOT}" \
  --output_dir "{RESULTS}/train_quex_eval" \
  --checkpoint_path "{RESULTS}/train_quex/best.pth" \
  --gpu "0" \
  --test_batch_size 128 \
  --num_workers 8 \
  --eval_only \
  --export_onnx \
  --compare_onnx \
  --onnx_runtime_provider cuda \
  --onnx_batch_size 0 \
  --benchmark_warmup_batches 2 \
  --grad_cam \
  --grad_cam_per_class \
  --grad_cam_per_class_samples 1


Loaded 27132 COVID-QU-Ex train images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-QU-EX/Lung Segmentation Data/Lung Segmentation Data
Loaded 6788 COVID-QU-Ex test images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-QU-EX/Lung Segmentation Data/Lung Segmentation Data
  + Number of Encoder Params: 14.77(e6)
  + Number of Network Params: 14.77(e6)
Loaded model weights from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/train_quex/best.pth
eval: 100%|█████████████████████████████████████| 54/54 [00:12<00:00,  4.45it/s]
accuracy=0.958603 | macro_auc=0.995027 | macro_ap=0.989962
/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/Downstream/2D/Cls/main_flexible.py:250: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about

## 5c. Cross-eval: model QU-Ex → COVID-19 Radiography

In [ ]:
!python Downstream/2D/Cls/main_flexible.py \
  --task covid \
  --covid_mode other_normal \
  --covid_root "{COVID_ROOT}" \
  --output_dir "{RESULTS}/eval_quex_to_covid19" \
  --checkpoint_path "{RESULTS}/train_quex/best.pth" \
  --gpu "0" \
  --test_batch_size 128 \
  --num_workers 8 \
  --eval_only


Loaded 16933 COVID other-normal train images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset
Loaded 4232 COVID other-normal test images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/COVID-19_Radiography_Dataset
  + Number of Encoder Params: 14.77(e6)
  + Number of Network Params: 14.77(e6)
Loaded model weights from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/train_quex/best.pth
eval: 100%|█████████████████████████████████████| 34/34 [00:08<00:00,  4.19it/s]
accuracy=0.961248 | macro_auc=0.996222 | macro_ap=0.994143


---
# Train NIH ChestX-ray14

## 6a. Train — 3 seeds

In [6]:
NIH_ROOT       = "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/ChestX-ray"
NIH_CSV        = f"{NIH_ROOT}/Data_Entry_2017.csv"
NIH_TRAIN_LIST = f"{NIH_ROOT}/train_val_list.txt"
NIH_TEST_LIST  = f"{NIH_ROOT}/test_list.txt"

import os
print("NIH root   :", os.path.exists(NIH_ROOT))
print("CSV        :", os.path.exists(NIH_CSV))
print("Train list :", os.path.exists(NIH_TRAIN_LIST))
print("Test list  :", os.path.exists(NIH_TEST_LIST))


NIH root   : True
CSV        : True
Train list : True
Test list  : True


In [6]:
!python Downstream/2D/Cls/main_flexible.py \
  --task nih \
  --nih_root "{NIH_ROOT}" \
  --nih_csv "{NIH_CSV}" \
  --nih_train_list "{NIH_TRAIN_LIST}" \
  --nih_test_list "{NIH_TEST_LIST}" \
  --output_dir "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/unimiss_nih" \
  --pre_train \
  --pre_train_path "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/UniMissPlus.pth" \
  --gpu "0" \
  --batch_size 128 \
  --test_batch_size 128 \
  --num_workers 8 \
  --epochs 30 \
  --learning_rate 0.0001 \
  --seeds "42"


=== Running seed 42 -> /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/unimiss_nih/seed_42 ===
Loaded 86524 NIH train images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/ChestX-ray
Loaded 25596 NIH test images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/ChestX-ray
  + Number of Encoder Params: 14.78(e6)
  + Number of Network Params: 14.78(e6)
Loaded 198 pretrained layers from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/UniMissPlus.pth
eval: 100%|███████████████████████████████████| 200/200 [03:43<00:00,  1.12s/it]
epoch=0 | train_loss=0.193804 | lr=9.69994e-05 | mean_auc=0.653189 | mean_ap=0.159152 | micro_f1=0.17411 | exact_match_accuracy=0.145179 | label_accuracy=0.907858
eval: 100%|███████████████████████████████████| 200/200 [03:20<00:00,  1.00s/it]
epoch=1 | train_loss=0.184388 | lr=9.3984e-05 | mean_auc=0.698281 | mean_ap=0.18059 | micro_f1=

## 6b. Eval + Export ONNX + Grad-CAM

In [10]:
!python Downstream/2D/Cls/main_flexible.py \
  --task nih \
  --nih_root "{NIH_ROOT}" \
  --nih_csv "{NIH_CSV}" \
  --nih_train_list "{NIH_TRAIN_LIST}" \
  --nih_test_list "{NIH_TEST_LIST}" \
  --output_dir "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/unimiss_nih_eval" \
  --checkpoint_path "/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/unimiss_nih/best.pth" \
  --gpu "0" \
  --test_batch_size 128 \
  --num_workers 8 \
  --eval_only \
  --export_onnx \
  --compare_onnx \
  --onnx_runtime_provider cuda \
  --onnx_batch_size 0 \
  --benchmark_warmup_batches 2 \
  --grad_cam \
  --grad_cam_samples 8


Loaded 86524 NIH train images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/ChestX-ray
Loaded 25596 NIH test images from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/dataset/ChestX-ray
  + Number of Encoder Params: 14.78(e6)
  + Number of Network Params: 14.78(e6)
Loaded model weights from /media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/results/unimiss_nih/best.pth
eval: 100%|███████████████████████████████████| 200/200 [00:49<00:00,  4.03it/s]
mean_auc=0.798795 | mean_ap=0.291649 | micro_f1=0.349225 | exact_match_accuracy=0.248359 | label_accuracy=0.91183
/media/pc/fa8cd839-121b-4df4-9c0c-0958d9216d28/UniMiSS-code/UniMiSSPlus/Downstream/2D/Cls/main_flexible.py:250: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/